# Neural Style Transfer
## Applying Artistic Styles to Photographs

This notebook demonstrates Neural Style Transfer using a pre-trained VGG19 model to transform regular images into stylized artistic versions. The technique combines content and style features extracted from deep convolutional neural networks.

## Section 1: Import Required Libraries

Import necessary libraries including PyTorch, torchvision, NumPy, Matplotlib, PIL for image processing, and deep learning.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Variable
import torch.nn.functional as F

import torchvision.transforms as transforms
from torchvision import models

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import urllib.request
from io import BytesIO
import os

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Section 2: Load Pre-trained VGG19 Model

Load a pre-trained VGG19 model from torchvision.models. We'll remove the classification layers and use it for feature extraction only.

In [ ]:
# Load pre-trained VGG19 model
vgg19 = models.vgg19(pretrained=True).to(device)

# Freeze all VGG19 parameters
for param in vgg19.parameters():
    param.requires_grad = False

# Extract only the feature layers (remove classifier)
cnn = vgg19.features

print("VGG19 Model loaded successfully!")
print(f"Number of layers: {len(list(cnn.children()))}")

## Section 3: Define Style Transfer Functions

Create helper functions for image loading, preprocessing, deprocessing, and normalization using ImageNet statistics.

In [ ]:
# ImageNet normalization statistics
imnet_mean = torch.tensor([0.485, 0.456, 0.406]).to(device)
imnet_std = torch.tensor([0.229, 0.224, 0.225]).to(device)

def load_image(image_path, max_size=512):
    """Load and preprocess image"""
    if isinstance(image_path, str):
        image = Image.open(image_path)
    else:
        # Handle URL or file-like object
        image = Image.open(BytesIO(image_path)).convert('RGB')
    
    # Resize image
    if max(image.size) > max_size:
        image.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
    
    return image

def preprocess_image(image, size=None):
    """Convert PIL image to tensor and normalize"""
    if size:
        image = image.resize((size, size), Image.Resampling.LANCZOS)
    
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=imnet_mean.cpu(), std=imnet_std.cpu())
    ])
    
    image = transform(image).unsqueeze(0)
    return image.to(device)

def deprocess_image(tensor):
    """Convert normalized tensor back to PIL image"""
    tensor = tensor.cpu().clone().squeeze(0)
    
    # Denormalize
    tensor[0] = tensor[0] * imnet_std[0] + imnet_mean[0]
    tensor[1] = tensor[1] * imnet_std[1] + imnet_mean[1]
    tensor[2] = tensor[2] * imnet_std[2] + imnet_mean[2]
    
    # Clip values to valid range
    tensor = tensor.clamp(0, 1)
    
    transform = transforms.ToPILImage()
    return transform(tensor)

print("Helper functions defined successfully!")

## Section 4: Load Content and Style Images

Load content image (photograph) and style image (artistic image). We'll create sample images or use images from the workspace.

In [ ]:
# Create a sample content image (simple photograph)
def create_content_image(size=256):
    """Create a simple content image with geometric shapes"""
    img = Image.new('RGB', (size, size), color='white')
    pixels = img.load()
    
    # Add some simple patterns
    for i in range(size):
        for j in range(size):
            if (i // 50 + j // 50) % 2 == 0:
                pixels[i, j] = (100 + i % 100, 150, 200 - j % 100)
            else:
                pixels[i, j] = (200 + i % 50, 100, 150)
    
    return img

# Create a sample style image (artistic style)
def create_style_image(size=256):
    """Create a simple style image with brush stroke pattern"""
    img = Image.new('RGB', (size, size), color='white')
    pixels = img.load()
    
    # Add stylized patterns (Van Gogh-like swirls)
    for i in range(size):
        for j in range(size):
            freq = np.sin(i / 30) * np.cos(j / 30)
            r = int(150 + 100 * np.sin(freq))
            g = int(100 + 100 * np.cos(freq * 2))
            b = int(200 + 50 * np.sin(freq * 3))
            pixels[i, j] = (max(0, min(255, r)), max(0, min(255, g)), max(0, min(255, b)))
    
    return img

# Create sample images
content_image = create_content_image(256)
style_image = create_style_image(256)

# Save sample images
os.makedirs('style_transfer_results', exist_ok=True)
content_image.save('style_transfer_results/content.jpg')
style_image.save('style_transfer_results/style.jpg')

# Display original images
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(content_image)
axes[0].set_title('Content Image (Photograph)')
axes[0].axis('off')

axes[1].imshow(style_image)
axes[1].set_title('Style Image (Artistic)')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('style_transfer_results/01_original_images.png', dpi=100, bbox_inches='tight')
plt.show()

# Preprocess images
content_tensor = preprocess_image(content_image, 256)
style_tensor = preprocess_image(style_image, 256)

print(f"Content tensor shape: {content_tensor.shape}")
print(f"Style tensor shape: {style_tensor.shape}")

## Section 5: Extract Feature Maps

Extract feature maps from intermediate layers of VGG19. We'll use conv_4 for content and conv_1 through conv_5 for style.

In [ ]:
# Define layers to extract features from
# VGG19 structure: relu1_1, relu1_2, pool1, relu2_1, relu2_2, pool2, ...
# We use relu layers for feature extraction

content_layers = ['relu4_2']  # Content representation layer
style_layers = ['relu1_1', 'relu2_1', 'relu3_1', 'relu4_1', 'relu5_1']  # Style representation layers

# Mapping of layer indices in VGG19 features
layer_mapping = {
    'relu1_1': 1,
    'relu1_2': 3,
    'relu2_1': 6,
    'relu2_2': 8,
    'relu3_1': 11,
    'relu3_2': 13,
    'relu4_1': 18,
    'relu4_2': 20,
    'relu5_1': 25,
    'relu5_2': 27
}

def get_feature_maps(image, layers_list):
    """Extract feature maps from specified layers"""
    features = {}
    x = image
    layer_names = list(layer_mapping.keys())
    
    for idx, layer in enumerate(cnn):
        x = layer(x)
        layer_name = layer_names[idx] if idx < len(layer_names) else None
        
        # Check if this layer's index matches our target layers
        for layer_str in layers_list:
            if layer_mapping[layer_str] == idx:
                features[layer_str] = x.detach()
    
    return features

# Extract features
content_features = get_feature_maps(content_tensor, content_layers)
style_features = get_feature_maps(style_tensor, style_layers)

print("Content features extracted from layers:", list(content_features.keys()))
print("Style features extracted from layers:", list(style_features.keys()))
print("\nFeature shapes:")
for layer, feature in content_features.items():
    print(f"  {layer}: {feature.shape}")

## Section 6: Calculate Loss Functions

Implement content loss (MSE between content features) and style loss (Gram matrix comparison). Combine losses with a style-to-content weight ratio.

In [ ]:
def gram_matrix(tensor):
    """Calculate Gram matrix for style loss"""
    b, c, h, w = tensor.size()
    
    # Reshape to (c, h*w)
    tensor = tensor.view(b, c, h * w)
    
    # Calculate Gram matrix: (c, h*w) @ (h*w, c) = (c, c)
    gram = torch.matmul(tensor, tensor.transpose(1, 2))
    
    return gram.div(c * h * w)

def content_loss(generated_features, content_features):
    """Calculate content loss (MSE between features)"""
    loss = 0
    for layer in content_layers:
        loss += F.mse_loss(generated_features[layer], content_features[layer])
    return loss

def style_loss(generated_features, style_features):
    """Calculate style loss (difference between Gram matrices)"""
    loss = 0
    for layer in style_layers:
        gen_gram = gram_matrix(generated_features[layer])
        style_gram = gram_matrix(style_features[layer])
        loss += F.mse_loss(gen_gram, style_gram)
    return loss

def total_loss(generated, content_feat, style_feat, content_weight=1.0, style_weight=100000.0):
    """Calculate total loss"""
    gen_features = get_feature_maps(generated, content_layers + style_layers)
    
    c_loss = content_loss(gen_features, content_feat)
    s_loss = style_loss(gen_features, style_feat)
    
    total = content_weight * c_loss + style_weight * s_loss
    return total, c_loss, s_loss

print("Loss functions defined successfully!")

## Section 7: Optimize Generated Image

Initialize generated image and use L-BFGS optimizer to minimize total loss. Update the generated image iteratively while displaying progress.

In [ ]:
# Initialize generated image as clone of content image
generated = content_tensor.clone().detach().requires_grad_(True)

# Use Adam optimizer (faster convergence than L-BFGS for this demonstration)
optimizer = optim.Adam([generated], lr=0.01)

# Optimization parameters
num_iterations = 200
content_weight = 1.0
style_weight = 100000.0

# Store losses for visualization
losses_history = {'total': [], 'content': [], 'style': []}

print(f"Starting Neural Style Transfer optimization for {num_iterations} iterations...")
print("=" * 60)

# Optimization loop
for iteration in range(num_iterations):
    optimizer.zero_grad()
    
    # Calculate losses
    total, c_loss, s_loss = total_loss(generated, content_features, style_features, 
                                        content_weight, style_weight)
    
    # Backpropagation
    total.backward()
    optimizer.step()
    
    # Clamp generated image to valid range
    with torch.no_grad():
        generated.clamp_(-2, 2)
    
    # Store losses
    losses_history['total'].append(total.item())
    losses_history['content'].append(c_loss.item())
    losses_history['style'].append(s_loss.item())
    
    # Print progress
    if (iteration + 1) % 20 == 0:
        print(f"Iteration {iteration + 1}/{num_iterations}")
        print(f"  Total Loss: {total.item():.4f}")
        print(f"  Content Loss: {c_loss.item():.6f}")
        print(f"  Style Loss: {s_loss.item():.4f}")
        print()

print("=" * 60)
print("Optimization completed!")

# Plot loss convergence
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(losses_history['total'], label='Total Loss')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss')
axes[0].set_title('Total Loss Convergence')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(losses_history['content'], label='Content Loss', color='green')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Loss')
axes[1].set_title('Content Loss Convergence')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

axes[2].plot(losses_history['style'], label='Style Loss', color='red')
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('Loss')
axes[2].set_title('Style Loss Convergence')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.savefig('style_transfer_results/02_loss_convergence.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nLoss convergence plot saved!")

## Section 8: Display Styled Results

Visualize and compare original content image, style image, and final styled output. Save the generated stylized image.

In [ ]:
# Convert generated tensor back to image
styled_image = deprocess_image(generated)

# Save the styled image
styled_image.save('style_transfer_results/styled_output.jpg')
print("Styled image saved to 'style_transfer_results/styled_output.jpg'")

# Create comparison visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Content image
axes[0].imshow(content_image)
axes[0].set_title('Content Image\n(Original Photograph)', fontsize=12, fontweight='bold')
axes[0].axis('off')

# Style image
axes[1].imshow(style_image)
axes[1].set_title('Style Image\n(Artistic Reference)', fontsize=12, fontweight='bold')
axes[1].axis('off')

# Styled output
axes[2].imshow(styled_image)
axes[2].set_title('Stylized Output\n(Result)', fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('style_transfer_results/03_final_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n" + "=" * 60)
print("NEURAL STYLE TRANSFER COMPLETE!")
print("=" * 60)
print(f"\nResults saved in 'style_transfer_results/' directory:")
print("  - 01_original_images.png: Content and style images")
print("  - 02_loss_convergence.png: Loss curves during optimization")
print("  - 03_final_comparison.png: Side-by-side comparison")
print("  - styled_output.jpg: Final styled image")

## Bonus: Advanced Style Transfer Techniques

### Trying Different Style-Content Weight Ratios

Explore how different weight ratios affect the balance between content preservation and style application.

In [ ]:
# Try different style/content weight ratios
def apply_style_transfer(content_img, style_img, iterations=150, 
                        content_weight=1.0, style_weight=100000.0):
    """Apply neural style transfer with custom parameters"""
    
    # Preprocess
    content_tensor_t = preprocess_image(content_img, 256)
    style_tensor_t = preprocess_image(style_img, 256)
    
    # Extract features
    content_features_t = get_feature_maps(content_tensor_t, content_layers)
    style_features_t = get_feature_maps(style_tensor_t, style_layers)
    
    # Initialize generated image
    generated_t = content_tensor_t.clone().detach().requires_grad_(True)
    optimizer_t = optim.Adam([generated_t], lr=0.01)
    
    # Optimization loop
    for iteration in range(iterations):
        optimizer_t.zero_grad()
        total_t, _, _ = total_loss(generated_t, content_features_t, style_features_t, 
                                   content_weight, style_weight)
        total_t.backward()
        optimizer_t.step()
        
        with torch.no_grad():
            generated_t.clamp_(-2, 2)
    
    return deprocess_image(generated_t)

# Apply with different weight ratios
print("Generating style transfer results with different weight ratios...")
print("This may take a moment...")

weight_ratios = [10000.0, 100000.0, 1000000.0]
results = []

for i, style_w in enumerate(weight_ratios, 1):
    print(f"Processing {i}/3 (style_weight={style_w})...")
    result = apply_style_transfer(content_image, style_image, 
                                 iterations=100, 
                                 content_weight=1.0, 
                                 style_weight=style_w)
    results.append(result)

# Visualize different weight ratios
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].imshow(content_image)
axes[0].set_title('Original Content', fontweight='bold')
axes[0].axis('off')

for i, (result, weight) in enumerate(zip(results, weight_ratios)):
    axes[i+1].imshow(result)
    axes[i+1].set_title(f'Style Weight: {weight:,.0f}', fontweight='bold')
    axes[i+1].axis('off')

plt.tight_layout()
plt.savefig('style_transfer_results/04_weight_ratios_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("Weight ratio comparison saved!")
print("\nKey Observations:")
print("- Lower style weight: More preservation of content")
print("- Higher style weight: Stronger artistic style application")

## Key Concepts Summary

### Neural Style Transfer Overview

**What is Neural Style Transfer?**
Neural Style Transfer is a technique that separates and recombines the content and style of images using deep convolutional neural networks. It uses two images:
- **Content Image**: The photograph we want to stylize
- **Style Image**: The artistic style we want to apply

### Core Components

**1. Feature Extraction (VGG19)**
- Uses pre-trained VGG19 convolutional layers to extract features
- Content features: Captured from deeper layers (e.g., relu4_2)
- Style features: Captured from multiple layers (relu1_1 through relu5_1)

**2. Loss Functions**

**Content Loss:** Measures how much the generated image's content differs from the original
```
L_content = MSE(generated_features, content_features)
```

**Style Loss:** Uses Gram matrices to capture texture and pattern information
```
L_style = Σ MSE(Gram(generated), Gram(style))
```

**Total Loss:** Weighted combination of both
```
L_total = α * L_content + β * L_style
```

**3. Optimization**
- Iteratively updates the generated image to minimize total loss
- Uses Adam optimizer for efficient convergence
- No neural network weights are updated, only the generated image

### Parameters & Their Effects

| Parameter | Effect |
|-----------|--------|
| **Content Weight** | Higher = More faithful to original content |
| **Style Weight** | Higher = Stronger artistic style application |
| **Iterations** | More = Better quality but longer computation |
| **Layer Selection** | Different layers capture different levels of abstraction |

### Advantages & Limitations

**Advantages:**
✓ Creates visually artistic results
✓ Works with any artistic style image
✓ Interpretable loss functions
✓ Uses pre-trained models (no training needed)

**Limitations:**
✗ Slow optimization (minutes per image)
✗ Content/style weight tuning is trial-and-error
✗ May fail with very different image resolutions
✗ Requires GPU for reasonable performance

### Real-world Applications

- Photo artistic rendering
- Creative content generation
- Video frame stylization
- Artistic filter applications
- Game asset generation

## Setup Instructions

### Required Dependencies

To run this notebook, install the following packages:

```bash
pip install torch torchvision pillow numpy matplotlib
```

### Running the Notebook

1. **Install Requirements**: Execute the commands above in your terminal
2. **Run Cells Sequentially**: Execute each cell from top to bottom
3. **GPU Acceleration** (Optional): If you have CUDA installed, PyTorch will automatically use your GPU for faster processing
4. **Output Directory**: All results are saved in the `style_transfer_results/` directory

### Performance Tips

- **GPU Usage**: Processing with GPU is ~10-100x faster than CPU
- **Image Size**: Larger images take longer; 256x256 is a good balance
- **Iterations**: 100-200 iterations usually produce good results
- **Memory**: Ensure sufficient GPU/RAM; reduce image size if out of memory

### Next Steps

1. Experiment with different content and style images
2. Adjust weight ratios to control the style/content balance
3. Try different layer selections for alternative effects
4. Combine with other techniques for advanced applications